# 08_sensitivity — Preprocessing sensitivity: the head-motion-corrected pipeline

**Manuscript:** Supplementary S2 (tab:motion_arms, tab:interp_arms, session-2 endpoints), S13 sign stability, S18 (tab:color_specificity, head-motion column).

Every neural endpoint of session 1 was recomputed on the head-motion-corrected pipeline (`00_preprocessing/scripts/hmc_v2/`) with every other element held fixed: the control interpolation gate (`perm_adjacent_arm.py`), eight-way classification (`loro_eightway_arm.py`), Procrustes disparity under both estimators (`loso_arm_wrapper.py` driving `02_geometry/scripts/rerun_loo_consistent.py`), the colour-correspondence permutation, and the session-2 endpoints (`exp2_endpoints_arms.json`). The manuscript reports both pipelines side by side and treats individual cells as descriptive.

**How to read this notebook.** Every code cell loads committed result files from `results/` and compares the values it derives with the numbers printed in the manuscript (`V.check`). A check passes when the produced value equals the printed one at the printed precision, or satisfies the stated relation. Quantities that have no committed artifact are recorded as pointers (`V.flag`) rather than silently omitted. The last cell tallies the checks and writes `_checks_08_sensitivity.json`, which `run_notebooks.py` collects into `REPORT.md`.

Provenance: built by `tools/public_repo/build.py` of the development repository (commit 53c81c2); manuscript source in `../paper/`; check list in `../MANIFEST.md`; code map in `../MAP.md`.

**Source and code map**

| Result file | Producing script | What it holds |
|---|---|---|
| `results/loso_two_arm_summary.json` | `scripts/loso_arm_wrapper.py -> ../02_geometry/scripts/rerun_loo_consistent.py` | Crawford-Howell and LOSO disparity on both pipelines (tab:motion_arms) |
| `results/perm_adjacent_arm_{primary,head_motion_correction}.json + null arrays` | `scripts/perm_adjacent_arm.py` | control interpolation gate and single-case hV4 contrasts per pipeline (tab:interp_arms) |
| `results/loro_eightway_arms.json` | `scripts/loro_eightway_arm.py` | LORO eight-way accuracy per pipeline with two-tailed single-case tests (tab:interp_arms) |
| `results/disparity_frozen_permutation_head_motion_correction.json` | `../02_geometry/scripts/color_specificity/disparity_frozen_permutation.py` | colour-correspondence grid, head-motion column of tab:color_specificity |
| `results/exp2_endpoints_arms.json` | `(exp2 pipeline rerun on the harmonised + head-motion-corrected arm)` | session-2 endpoints under both arms (S2) |
| `results/disparity_individual_arms.json` | `(frozen wrapper stdout, parsed)` | disparity per arm, descriptive |

In [1]:
import sys, json, csv
from pathlib import Path
sys.path.insert(0, str((Path.cwd() / ".." / "common").resolve()))
import numpy as np
from scipy import stats
import verify as V
from stats_helpers import crawford_howell, hedges_g, bh_fdr, wilson_interval
R = Path("results")
def J(name):
    with open(R / name) as f:
        return json.load(f)
HC = [f"sub-{i:02d}" for i in range(1, 8)]
CVD = {"deutan": "sub-08", "protan": "sub-09"}
ROIS = ["V1", "V2", "V3", "hV4"]
HUES = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "magenta"]

ls = J("loso_two_arm_summary.json")["arms"]
pa = {"primary": J("perm_adjacent_arm_primary.json"), "hmc": J("perm_adjacent_arm_head_motion_correction.json")}
le = J("loro_eightway_arms.json")["arms"]

V.start("08_sensitivity")

### Disparity under head-motion correction (Supplementary S2, tab:motion_arms)
Crawford-Howell one-tailed t (p) and the leave-one-subject-out estimator, head-motion-corrected pipeline.

| id | manuscript | quantity | reported |
|---|---|---|---|
| 08.01 | tab:motion_arms | 8 cells x (CH t, p; LOSO t, p) | `32 cells, see 08.T1.*` |
| 08.02 | S2 'Disparity under the two pipelines' | protan V1 remains the largest deviation under both pipelines | `('V1', 'V1')` |
| 08.03 | S2 | protan V1 weakens from p = .007 | `0.007` |
| 08.04 | S2 | to p = .077 | `0.077` |
| 08.05 | S2 | deutan largest deviation moves from V2 to V1 | `('V2', 'V1')` |
| 08.06 | S2 | deutan V2 t reverses in sign (+2.1 -> -1.0) | `(2.1, -1.0)` |
| 08.07 | S2 | under LOSO the primary-pipeline protan V1 cell alone reaches significance across both pipelines | `[('with_residuals', 'sub-09', 'V1')]` |
| 08.08 | S2 | primary-pipeline table reproduces tab:disparity_loso (deutan V2 .040 / .116; protan V1 .007 / .045) | `(0.04, 0.116, 0.007, 0.045)` |

In [2]:
T = {("sub-08", "V1"): (2.4, 0.027, 1.6, 0.080), ("sub-08", "V2"): (-1.0, 0.825, -0.6, 0.723), ("sub-08", "V3"): (0.6, 0.293, 0.4, 0.338), ("sub-08", "hV4"): (0.4, 0.351, 0.4, 0.356),
     ("sub-09", "V1"): (1.6, 0.077, 1.3, 0.121), ("sub-09", "V2"): (1.0, 0.186, 0.1, 0.480), ("sub-09", "V3"): (1.1, 0.151, 1.0, 0.178), ("sub-09", "hV4"): (1.4, 0.101, 1.1, 0.159)}
name = {"sub-08": "Deutan", "sub-09": "Protan"}
for (s, roi), (t_c, p_c, t_l, p_l) in T.items():
    c = ls["hmc_v2"][roi][s]
    V.check(f"08.T1.{name[s]}.{roi}.t", f"tab:motion_arms {name[s]} {roi} CH t", c["ch"]["t"], t_c, nd=1)
    V.check(f"08.T1.{name[s]}.{roi}.p", f"tab:motion_arms {name[s]} {roi} CH p", c["ch"]["p"], p_c, nd=3)
    V.check(f"08.T1.{name[s]}.{roi}.t_loso", f"tab:motion_arms {name[s]} {roi} LOSO t", c["loso"]["t"], t_l, nd=1)
    V.check(f"08.T1.{name[s]}.{roi}.p_loso", f"tab:motion_arms {name[s]} {roi} LOSO p", c["loso"]["p"], p_l, nd=3)
prim = ls["with_residuals"]
max_roi = {arm: {s: max(ROIS, key=lambda r: ls[arm][r][s]["ch"]["t"]) for s in ("sub-08", "sub-09")} for arm in ls}
sig_loso = [(arm, s, r) for arm in ls for s in ("sub-08", "sub-09") for r in ROIS if ls[arm][r][s]["loso"]["p"] < 0.05]
print(max_roi, sig_loso, prim["V1"]["sub-09"]["ch"]["p"], ls["hmc_v2"]["V1"]["sub-09"]["ch"]["p"], prim["V2"]["sub-08"]["ch"]["t"], ls["hmc_v2"]["V2"]["sub-08"]["ch"]["t"])
V.table('08.01', 'tab:motion_arms | 8 cells x (CH t, p; LOSO t, p)', '32 cells, see 08.T1.*')
V.check('08.02', "S2 'Disparity under the two pipelines' | protan V1 remains the largest deviation under both pipelines", (max_roi["with_residuals"]["sub-09"], max_roi["hmc_v2"]["sub-09"]), ('V1', 'V1'), mode='eq')
V.check('08.03', 'S2 | protan V1 weakens from p = .007', prim["V1"]["sub-09"]["ch"]["p"], 0.007, nd=3)
V.check('08.04', 'S2 | to p = .077', ls["hmc_v2"]["V1"]["sub-09"]["ch"]["p"], 0.077, nd=3)
V.check('08.05', 'S2 | deutan largest deviation moves from V2 to V1', (max_roi["with_residuals"]["sub-08"], max_roi["hmc_v2"]["sub-08"]), ('V2', 'V1'), mode='eq')
V.check('08.06', 'S2 | deutan V2 t reverses in sign (+2.1 -> -1.0)', (round(prim["V2"]["sub-08"]["ch"]["t"], 1), round(ls["hmc_v2"]["V2"]["sub-08"]["ch"]["t"], 1)), (2.1, -1.0), mode='eq')
V.check('08.07', 'S2 | under LOSO the primary-pipeline protan V1 cell alone reaches significance across both pipelines', sig_loso, [('with_residuals', 'sub-09', 'V1')], mode='eq')
V.check('08.08', 'S2 | primary-pipeline table reproduces tab:disparity_loso (deutan V2 .040 / .116; protan V1 .007 / .045)', (round(prim["V2"]["sub-08"]["ch"]["p"], 3), round(prim["V2"]["sub-08"]["loso"]["p"], 3), round(prim["V1"]["sub-09"]["ch"]["p"], 3), round(prim["V1"]["sub-09"]["loso"]["p"], 3)), (0.04, 0.116, 0.007, 0.045), mode='eq')

[OK ] 08.T1.Deutan.V1.t tab:motion_arms Deutan V1 CH t: produced=2.402  reported=2.4
[OK ] 08.T1.Deutan.V1.p tab:motion_arms Deutan V1 CH p: produced=0.02656  reported=0.027
[OK ] 08.T1.Deutan.V1.t_loso tab:motion_arms Deutan V1 LOSO t: produced=1.607  reported=1.6
[OK ] 08.T1.Deutan.V1.p_loso tab:motion_arms Deutan V1 LOSO p: produced=0.07957  reported=0.08
[OK ] 08.T1.Deutan.V2.t tab:motion_arms Deutan V2 CH t: produced=-1.014  reported=-1
[OK ] 08.T1.Deutan.V2.p tab:motion_arms Deutan V2 CH p: produced=0.8251  reported=0.825
[OK ] 08.T1.Deutan.V2.t_loso tab:motion_arms Deutan V2 LOSO t: produced=-0.628  reported=-0.6
[OK ] 08.T1.Deutan.V2.p_loso tab:motion_arms Deutan V2 LOSO p: produced=0.7234  reported=0.723
[OK ] 08.T1.Deutan.V3.t tab:motion_arms Deutan V3 CH t: produced=0.5746  reported=0.6
[OK ] 08.T1.Deutan.V3.p tab:motion_arms Deutan V3 CH p: produced=0.2932  reported=0.293
[OK ] 08.T1.Deutan.V3.t_loso tab:motion_arms Deutan V3 LOSO t: produced=0.4383  reported=0.4
[OK ] 08.T

### Classification and interpolation under the two pipelines (Supplementary S2, tab:interp_arms)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 08.09 | tab:interp_arms | interpolation and classification cells under both pipelines | `34 cells, see 08.T2.*` |
| 08.10 | S2 'Classification and interpolation' | hV4 alone passes the control gate in both pipelines | `True` |
| 08.11 | S2 | both participants remain below the control mean in both pipelines | `True` |
| 08.12 | S2 | lowest CVD classification cell, primary | `0.375` |
| 08.13 | S2 | lowest CVD classification cell, head-motion correction | `0.229` |
| 08.14 | S2 | which is 1.8 times chance | `1.8` |
| 08.15 | S2 | every single-case classification contrast stays above p = .10 in both pipelines | `0.1` |
| 08.16 | S2 | single-case interpolation contrasts reach significance under the primary pipeline alone | `(True, True)` |
| 08.17 | tab:interp_arms caption | permutation null mean is 0.35 in both pipelines (all ROIs within 0.34-0.36) | `(0.34, 0.36)` |

In [3]:
T_CTRL = {"V1": (0.393, 0.164, 0.283, 0.922), "V2": (0.357, 0.424, 0.381, 0.228), "V3": (0.339, 0.586, 0.315, 0.810), "hV4": (0.456, 0.011, 0.451, 0.023)}
for roi, (a_p, p_p, a_h, p_h) in T_CTRL.items():
    V.check(f"08.T2.controls.{roi}.primary.acc", f"tab:interp_arms controls {roi} primary", pa["primary"][roi]["observed"], a_p, nd=3)
    V.check(f"08.T2.controls.{roi}.primary.p", f"tab:interp_arms controls {roi} primary p", pa["primary"][roi]["p_perm"], p_p, nd=3)
    V.check(f"08.T2.controls.{roi}.hmc.acc", f"tab:interp_arms controls {roi} head-motion", pa["hmc"][roi]["observed"], a_h, nd=3)
    V.check(f"08.T2.controls.{roi}.hmc.p", f"tab:interp_arms controls {roi} head-motion p", pa["hmc"][roi]["p_perm"], p_h, nd=3)
T_CVD = {"deutan": (0.250, 0.054, 0.354, 0.242), "protan": (0.125, 0.012, 0.271, 0.108)}
for k, (a_p, p_p, a_h, p_h) in T_CVD.items():
    V.check(f"08.T2.{k}.hV4.primary.acc", f"tab:interp_arms {k} hV4 primary", pa["primary"]["hV4"]["cvd"][k]["adjacent"], a_p, nd=3)
    V.check(f"08.T2.{k}.hV4.primary.p", f"tab:interp_arms {k} hV4 primary p", pa["primary"]["hV4"]["cvd"][k]["p_one_tailed_lower"], p_p, nd=3)
    V.check(f"08.T2.{k}.hV4.hmc.acc", f"tab:interp_arms {k} hV4 head-motion", pa["hmc"]["hV4"]["cvd"][k]["adjacent"], a_h, nd=3)
    V.check(f"08.T2.{k}.hV4.hmc.p", f"tab:interp_arms {k} hV4 head-motion p", pa["hmc"]["hV4"]["cvd"][k]["p_one_tailed_lower"], p_h, nd=3)
T_LORO = {"controls": (0.488, None, 0.569, None), "deutan": (0.375, 0.351, 0.583, 0.905), "protan": (0.375, 0.351, 0.396, 0.197)}
for k, (a_p, p_p, a_h, p_h) in T_LORO.items():
    for arm, a_r, p_r in (("Primary", a_p, p_p), ("Realignment", a_h, p_h)):
        blk = le[arm]["hV4"]
        if k == "controls":
            V.check(f"08.T2.loro.controls.hV4.{arm}", f"tab:interp_arms classification controls hV4 {arm}", blk["hc_mean"], a_r, nd=3)
        else:
            V.check(f"08.T2.loro.{k}.hV4.{arm}.acc", f"tab:interp_arms classification {k} hV4 {arm}", blk[k]["accuracy"], a_r, nd=3)
            V.check(f"08.T2.loro.{k}.hV4.{arm}.p", f"tab:interp_arms classification {k} hV4 {arm} p (two-tailed)", blk[k]["p_two_tailed"], p_r, nd=3)
gate_hv4_only = all((pa[a][r]["p_perm"] < 0.05) == (r == "hV4") for a in pa for r in ROIS)
below_mean_both = all(pa[a]["hV4"]["cvd"][k]["adjacent"] < pa[a]["hV4"]["observed"] for a in pa for k in ("deutan", "protan"))
min_loro = {arm: min(le[arm][r][k]["accuracy"] for r in ROIS for k in ("deutan", "protan")) for arm in le}
min_loro_p = min(le[arm][r][k]["p_two_tailed"] for arm in le for r in ROIS for k in ("deutan", "protan"))
null_means = [pa[a][r]["null_mean"] for a in pa for r in ROIS]
print(gate_hv4_only, below_mean_both, min_loro, min_loro_p, null_means)
V.table('08.09', 'tab:interp_arms | interpolation and classification cells under both pipelines', '34 cells, see 08.T2.*')
V.check('08.10', "S2 'Classification and interpolation' | hV4 alone passes the control gate in both pipelines", gate_hv4_only, True, mode='eq')
V.check('08.11', 'S2 | both participants remain below the control mean in both pipelines', below_mean_both, True, mode='eq')
V.check('08.12', 'S2 | lowest CVD classification cell, primary', min_loro["Primary"], 0.375, nd=3)
V.check('08.13', 'S2 | lowest CVD classification cell, head-motion correction', min_loro["Realignment"], 0.229, nd=3)
V.check('08.14', 'S2 | which is 1.8 times chance', min_loro["Realignment"] / 0.125, 1.8, nd=1)
V.check('08.15', 'S2 | every single-case classification contrast stays above p = .10 in both pipelines', min_loro_p, 0.1, mode='gt')
V.check('08.16', 'S2 | single-case interpolation contrasts reach significance under the primary pipeline alone', (pa["primary"]["hV4"]["cvd"]["protan"]["p_one_tailed_lower"] < 0.05, min(pa["hmc"]["hV4"]["cvd"][k]["p_one_tailed_lower"] for k in ("deutan", "protan")) > 0.10), (True, True), mode='eq')
V.check('08.17', 'tab:interp_arms caption | permutation null mean is 0.35 in both pipelines (all ROIs within 0.34-0.36)', (min(null_means), max(null_means)), (0.34, 0.36), mode='range')

[OK ] 08.T2.controls.V1.primary.acc tab:interp_arms controls V1 primary: produced=0.3929  reported=0.393
[OK ] 08.T2.controls.V1.primary.p tab:interp_arms controls V1 primary p: produced=0.1638  reported=0.164
[OK ] 08.T2.controls.V1.hmc.acc tab:interp_arms controls V1 head-motion: produced=0.2827  reported=0.283
[OK ] 08.T2.controls.V1.hmc.p tab:interp_arms controls V1 head-motion p: produced=0.9221  reported=0.922
[OK ] 08.T2.controls.V2.primary.acc tab:interp_arms controls V2 primary: produced=0.3571  reported=0.357
[OK ] 08.T2.controls.V2.primary.p tab:interp_arms controls V2 primary p: produced=0.4236  reported=0.424
[OK ] 08.T2.controls.V2.hmc.acc tab:interp_arms controls V2 head-motion: produced=0.381  reported=0.381
[OK ] 08.T2.controls.V2.hmc.p tab:interp_arms controls V2 head-motion p: produced=0.2278  reported=0.228
[OK ] 08.T2.controls.V3.primary.acc tab:interp_arms controls V3 primary: produced=0.3393  reported=0.339
[OK ] 08.T2.controls.V3.primary.p tab:interp_arms contro

### Colour specificity under head-motion correction (Supplementary S18, tab:color_specificity right block)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 08.18 | tab:color_specificity | 35 cells, head-motion-corrected pipeline | `35 cells, see 08.T3.*` |
| 08.19 | S18 ¶3 | cells below .05 under head-motion correction | `15` |
| 08.20 | S18 ¶3 | none survive BH correction | `0` |
| 08.21 | S18 ¶3 | deutan V2 nominal p .003 | `0.003` |
| 08.22 | S18 ¶3 | deutan V2 corrected q .053 | `0.053` |
| 08.23 | S18 ¶3 | protan V1 moves to p .010 | `0.01` |

In [4]:
fz = J("disparity_frozen_permutation_head_motion_correction.json")["results"]
T_CS = {"Control 1": (0.012, 0.072, 0.018, 0.290), "Control 2": (0.437, 0.420, 0.191, 0.019), "Control 3": (0.325, 0.113, 0.176, 0.101),
        "Control 4": (0.420, 0.006, 0.015, 0.173), "Control 5": (0.079, 0.194, 0.032, 0.038), "Control 6": (0.373, 0.117, 0.010, 0.012),
        "Control 7": (0.008, 0.005, 0.391, None), "Deutan": (0.022, 0.003, 0.047, 0.257), "Protan": (0.010, 0.148, 0.053, 0.815)}
SUBMAP = {f"Control {i}": f"sub-{i:02d}" for i in range(1, 8)}; SUBMAP.update({"Deutan": "sub-08", "Protan": "sub-09"})
cells = {}
for lab, vals in T_CS.items():
    s = SUBMAP[lab]
    for roi, rep in zip(ROIS, vals):
        fr = fz[roi]["modes"]["frozen_projection"]; rec = fr["hc"]["per_subject"].get(s) or fr["cvd"].get(s)
        if rep is None:
            V.check(f"08.T3.{lab}.{roi}", f"tab:color_specificity head-motion {lab} {roi} (absent)", rec is None, True, mode="eq"); continue
        cells[(s, roi)] = rec["p_perm"]
        V.check(f"08.T3.{lab}.{roi}", f"tab:color_specificity head-motion {lab} {roi} p_perm", rec["p_perm"], rep, nd=3)
labels = list(cells); p_arr = np.array([cells[k] for k in labels]); q = dict(zip(labels, bh_fdr(p_arr)))
n_raw = int((p_arr < 0.05).sum()); n_bh = int(sum(v < 0.05 for v in q.values()))
print(n_raw, n_bh, q[("sub-08", "V2")])
V.table('08.18', 'tab:color_specificity | 35 cells, head-motion-corrected pipeline', '35 cells, see 08.T3.*')
V.check('08.19', 'S18 ¶3 | cells below .05 under head-motion correction', n_raw, 15, mode='eq')
V.check('08.20', 'S18 ¶3 | none survive BH correction', n_bh, 0, mode='eq')
V.check('08.21', 'S18 ¶3 | deutan V2 nominal p .003', cells[("sub-08", "V2")], 0.003, nd=3)
V.check('08.22', 'S18 ¶3 | deutan V2 corrected q .053', q[("sub-08", "V2")], 0.053, nd=3)
V.check('08.23', 'S18 ¶3 | protan V1 moves to p .010', cells[("sub-09", "V1")], 0.01, nd=3)

[OK ] 08.T3.Control 1.V1 tab:color_specificity head-motion Control 1 V1 p_perm: produced=0.012  reported=0.012
[OK ] 08.T3.Control 1.V2 tab:color_specificity head-motion Control 1 V2 p_perm: produced=0.0719  reported=0.072
[OK ] 08.T3.Control 1.V3 tab:color_specificity head-motion Control 1 V3 p_perm: produced=0.018  reported=0.018
[OK ] 08.T3.Control 1.hV4 tab:color_specificity head-motion Control 1 hV4 p_perm: produced=0.2897  reported=0.29
[OK ] 08.T3.Control 2.V1 tab:color_specificity head-motion Control 2 V1 p_perm: produced=0.4366  reported=0.437
[OK ] 08.T3.Control 2.V2 tab:color_specificity head-motion Control 2 V2 p_perm: produced=0.4196  reported=0.42
[OK ] 08.T3.Control 2.V3 tab:color_specificity head-motion Control 2 V3 p_perm: produced=0.1908  reported=0.191
[OK ] 08.T3.Control 2.hV4 tab:color_specificity head-motion Control 2 hV4 p_perm: produced=0.019  reported=0.019
[OK ] 08.T3.Control 3.V1 tab:color_specificity head-motion Control 3 V1 p_perm: produced=0.3247  reported

### Session-2 endpoints under the harmonised arm (Supplementary S2)

| id | manuscript | quantity | reported |
|---|---|---|---|
| 08.24 | S2 'Session-2 endpoints' | fourteen pre-specified endpoints | `14` |
| 08.25 | S2 | control reference hV4 adjacent accuracy, primary arm | `0.456` |
| 08.26 | S2 | harmonised arm | `0.445` |
| 08.27 | S2 | native-mask directional contrasts reversed between arms | `(8, 10)` |
| 08.28 | S2 | run-matched-mask contrasts reversed | `(5, 10)` |
| 08.29 | S2 | the contrasts at the pre-specified target regions held in both arms | `(True, True)` |

In [5]:
ea = J("exp2_endpoints_arms.json")
hc_ref = ea["variants"]["matched"]["hv4_adjacc"]
tgt = ea["variants"]["matched"]["direction_stability"]
tgt_keys = [k for k in tgt if "_V2_" in k or "_V1_" in k]
print(ea["prespecified_cells"], ea["flips_by_variant"], hc_ref["deutan_canon"]["hc_n4_mean"], hc_ref["deutan_harm"]["hc_n4_mean"], tgt_keys, [tgt[k]["flipped"] for k in tgt_keys])
V.check('08.24', "S2 'Session-2 endpoints' | fourteen pre-specified endpoints", ea["prespecified_cells"], 14, mode='eq')
V.check('08.25', 'S2 | control reference hV4 adjacent accuracy, primary arm', hc_ref["deutan_canon"]["hc_n4_mean"], 0.456, nd=3)
V.check('08.26', 'S2 | harmonised arm', hc_ref["deutan_harm"]["hc_n4_mean"], 0.445, nd=3)
V.check('08.27', 'S2 | native-mask directional contrasts reversed between arms', (ea["flips_by_variant"]["native"]["n_flipped"], ea["flips_by_variant"]["native"]["n_contrasts"]), (8, 10), mode='eq')
V.check('08.28', 'S2 | run-matched-mask contrasts reversed', (ea["flips_by_variant"]["matched"]["n_flipped"], ea["flips_by_variant"]["matched"]["n_contrasts"]), (5, 10), mode='eq')
V.check('08.29', 'S2 | the contrasts at the pre-specified target regions held in both arms', (len(tgt_keys) > 0, all(not tgt[k]["flipped"] for k in tgt_keys)), (True, True), mode='eq')

14 {'native': {'n_contrasts': 10, 'n_flipped': 8}, 'matched': {'n_contrasts': 10, 'n_flipped': 5}} 0.45555555555555555 0.4454861111111112 ['deutan_V2_opt_gt_win', 'protan_V1_opt_gt_win'] [False, False]
[OK ] 08.24 S2 'Session-2 endpoints' | fourteen pre-specified endpoints: produced=14  reported=14
[OK ] 08.25 S2 | control reference hV4 adjacent accuracy, primary arm: produced=0.4556  reported=0.456
[OK ] 08.26 S2 | harmonised arm: produced=0.4455  reported=0.445
[OK ] 08.27 S2 | native-mask directional contrasts reversed between arms: produced=(8, 10)  reported=(8, 10)
[OK ] 08.28 S2 | run-matched-mask contrasts reversed: produced=(5, 10)  reported=(5, 10)
[OK ] 08.29 S2 | the contrasts at the pre-specified target regions held in both arms: produced=(True, True)  reported=(True, True)


In [6]:
V.summary()


=== 08_sensitivity: 127/128 numeric checks reproduced exactly; 1 within one unit of the last printed digit; 0 mismatch, 0 error, 0 pointer-only ===
  NEAR     08.T2.protan.hV4.primary.p tab:interp_arms protan hV4 primary p: produced=0.01146 reported=0.012
